# Syon — Treinamento no Kaggle

1. **Settings → Accelerator → GPU T4 x2**
2. **Internet → ON**
3. Upload do projeto como Dataset Kaggle (zip da pasta Syon)
4. Output: `/kaggle/working/syon-output/syon-kaggle-lora/`

In [ ]:
# CELULA 1 — Copiar projeto Syon para /kaggle/working/syon
import os, sys, shutil
from pathlib import Path

SYON_SRC = Path("/kaggle/input/datasets/regyfelipe/syon-project/Syon")
PROJECT = Path("/kaggle/working/syon")

def ok(p):
    return p.is_dir() and (p/"pyproject.toml").exists() and (p/"training").is_dir()

# Fallback: busca automatica se o caminho mudar
if not ok(SYON_SRC):
    for script in Path("/kaggle/input").rglob("kaggle_train.py"):
        root = script.parent.parent.parent  # scripts/kaggle -> Syon/
        if ok(root):
            SYON_SRC = root
            break

if not ok(SYON_SRC):
    raise FileNotFoundError(f"Syon nao encontrado. Verifique Add Data. Tentou: {SYON_SRC}")

print(f"Origem: {SYON_SRC}")
if not PROJECT.exists():
    shutil.copytree(SYON_SRC, PROJECT)
os.chdir(PROJECT)
sys.path[:0] = [str(PROJECT), str(PROJECT / "src")]
print(f"OK -> {PROJECT}")
print(f"  training/: {(PROJECT/'training').exists()}")
print(f"  kaggle_train.py: {(PROJECT/'scripts'/'kaggle'/'kaggle_train.py').exists()}")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ Ative GPU em Settings → Accelerator")

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets pyyaml pydantic-settings safetensors

In [ ]:
# Preparar dataset (amostras demo + opcional /kaggle/input)
!python scripts/kaggle/prepare_dataset.py --data-dir /kaggle/working/syon/data/raw

In [ ]:
# Treinar (QLoRA + Security-Aware Loss — otimizado para T4 16GB)
!python scripts/kaggle/kaggle_train.py \
    --config training/configs/kaggle_config.yaml \
    --data-dir /kaggle/working/syon/data/raw \
    --max-steps 500

In [ ]:
# Verificar output
import json
from pathlib import Path

output = Path("/kaggle/working/syon-output")
if output.exists():
    summary = output / "training_summary.json"
    if summary.exists():
        print(json.dumps(json.loads(summary.read_text()), indent=2))
    print("Arquivos:", list(output.rglob("*"))[:20])
else:
    print("Output ainda não gerado")